# AgriWatch Monitor — anomalie suszy dla winnicy (Condom, Gers)
Otwieraj z GitHub (Plik → Otwórz → GitHub). Środowisko z **GPU** (dla SEN2SR). Potem `Uruchom wszystko`.

Łańcuch: Sentinel-2 (rastry, jak w v1) → SEN2SR 2,5 m z kontrolą → indeksy winnicy → anomalie (ERA5-Land SMA, SPI, NDVI)
→ status dekadowy wg logiki EDO CDI → walidacja na ISMN Condom → biuletyn i **dashboard winnicy**.
Wyniki: `MyDrive/GeoWorldLook/agriwatch/data/05_Final_Outputs/agriwatch/` i tabele `data/registry/gwl_*.csv`.


## 1. Dysk Google i kod z GitHub


In [1]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = 'https://github.com/geoworldlook/1_geoworldlook-agriscreen.git'
PROJECT_DIR = '/content/drive/MyDrive/GeoWorldLook/agriwatch'
if IN_COLAB:
    if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
        os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)
        !git clone {REPO_URL} "{PROJECT_DIR}"
    # Lokalne zmiany w kodzie (np. zapisany notatnik) blokowały `pull --ff-only` i kod zostawał stary:
    # odkładamy je do stash (do odzyskania: git stash list / git stash pop), potem pull.
    !git -C "{PROJECT_DIR}" stash push -q -m "colab-autostash" || true
    !git -C "{PROJECT_DIR}" pull --ff-only
    !git -C "{PROJECT_DIR}" log -1 --format="Kod: %h %ci %s"
    if os.popen(f'git -C "{PROJECT_DIR}" rev-parse HEAD').read() != os.popen(f'git -C "{PROJECT_DIR}" rev-parse origin/main').read():
        print('[UWAGA] Kod na Dysku różni się od origin/main — wyniki będą z innej wersji kodu.')
else:
    PROJECT_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import importlib
sys.modules.setdefault('imp', importlib)
try:
    %load_ext autoreload
    %autoreload 2
except Exception as e:
    print(f'[INFO] autoreload pominięty ({e})')
print('Projekt:', PROJECT_DIR)


Mounted at /content/drive
Already up to date.
Kod: 7f73dfb 2026-10-09 06:36:21 +0000 feat(weather): frost and heat hazards validated on Météo-France stations, surface soil layer, 5-year dashboard
Projekt: /content/drive/MyDrive/GeoWorldLook/agriwatch


## 2. Środowisko: pakiety, GEE, rejestr, konfiguracja


In [2]:
from step_05_colab_run import (setup_runtime, monitor_config, run_task, run_history, registry_summary,
                               task_ingest_s2, task_ingest_era5, task_scene_stats, task_anomalies,
                               task_validate, task_weather, task_bulletin, task_dashboard, task_summary)
from step_06_dashboard import show_dashboard
rt = setup_runtime(PROJECT_DIR, gee_project='ee-geoworldlook', pull=False)
cfg = monitor_config(rt)   # zmiany: monitor_config(rt, {'VINEYARDS': {'VINEYARD_06': 6, 'VINEYARD_07': 7}})



earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.). Uruchamianie procedury autoryzacji...
*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## 3. Dane: rastry Sentinel-2 (przyrostowo) i ERA5-Land 1991 → dziś


In [ ]:
run_task(rt, 'ingest_s2', task_ingest_s2, rt, cfg)
run_task(rt, 'ingest_era5', task_ingest_era5, rt, cfg)


[SKIPPED] ingest_s2 (7.2 s) 414 scen na Dysku, 0 nowych


## 4. Indeksy winnicy i stacji: 10 m oraz SEN2SR 2,5 m (kontrola H-SR0/H-SR1, bez zastępstwa interpolacją)


In [ ]:
run_task(rt, 'scene_stats', task_scene_stats, rt, cfg)


## 5. Anomalie, status, walidacja, przymrozki i upały, biuletyn
`weather`: dane dobowe Météo-France (Gers, Lot-et-Garonne) i ERA5-Land w punktach najbliższych stacji — walidacja
Tmin, Tmax, opadu i SPI oraz kalibracja progów przymrozku i upału. Pierwsze uruchomienie pobiera ERA5 dla stacji
(kilka minut), kolejne korzystają z cache.


In [ ]:
run_task(rt, 'anomalies', task_anomalies, rt, cfg)
run_task(rt, 'validate', task_validate, rt, cfg)
run_task(rt, 'weather', task_weather, rt, cfg)
run_task(rt, 'bulletin', task_bulletin, rt, cfg)


## 6. Dashboard winnicy
Mapa NDVI na zdjęciu satelitarnym (wybór daty), status, ryzyko suszy, przyczyny, pogoda i prognoza, wiarygodność.
Plik: `data/05_Final_Outputs/agriwatch/dashboard.html` (można otworzyć w przeglądarce z Dysku).


In [ ]:
run_task(rt, 'dashboard', task_dashboard, rt, cfg)
p = os.path.join(cfg['OUTPUT_DIR'], 'dashboard.html')
if os.path.exists(p):
    show_dashboard(p)
else:
    print('Dashboard nie powstał — sprawdź komunikat zadania powyżej.')


## 7. Raport z uruchomienia (do analizy)
`run_summary.md` — wersje, konfiguracja, pokrycie danych, kontrola SR, status per rok, walidacja, błędy.
**Po każdym większym uruchomieniu prześlij ten plik** (albo wklej jego treść) — na nim opieramy wnioski.


In [ ]:
run_task(rt, 'summary', task_summary, rt, cfg)
print(open(os.path.join(cfg['OUTPUT_DIR'], 'run_summary.md'), encoding='utf-8').read())


## 8. Rejestr i historia uruchomień


In [ ]:
display(registry_summary(rt))
display(run_history(rt, n=10))
